In [1]:
import sys, os
sys.path.append('./../')
import torch
import numpy as np
import time
from config import absjoin, PANDA_XML, SHELF_URDF, BASE_URDF


from envs.env_panda_shelf_gs import PandaShelf
 
from panda_kinematics import PandaKinematics


from src.ttts_augment import TTTS
from robot_utils import Point2PointMotion


%load_ext autoreload
%autoreload 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'

[I 07/03/25 11:27:20.625 182454] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
ARM_BASE_HEIGHT = 0.7

MAX_BATCH = 100

In [3]:
def env_infos_gen():
    """
    This function generates the environment information 
    useful for defining the simulation environment.
    """
    shelf_info = {
        "name": "shelf",
        "urdf": SHELF_URDF,
        "base_pose": [0.7, 0.0, 0.015, 0, 0, 90],
    }
    print("SHELF_URDF: ", SHELF_URDF)

    base_info = {
        "name": "base",
        "urdf": BASE_URDF,
        "base_pose": [0.0, 0.0, 0.35, 0, 0, 90]
    }

    robot_info = {
        "name": "panda",
        "xml": PANDA_XML,
        "base_pose": [0.0, 0.0, 0.8, 0, 0, 0],
        # "conf": [0.0, -np.pi/4, 0.0, -np.pi/2, 
        #          0, np.pi/2, np.pi/4, 0.06, 0.06],
        "conf": [1.76, -4.5e-01, -1.77, -1.35,  4.9e-01,
                2.36, -2.8e-01, 0.06, 0.06],
        "jnt_names": ["joint1", "joint2", "joint3",
                    "joint4", "joint5", "joint6",
                    "joint7", "finger_joint1", "finger_joint2", ],
        "kp": np.array([4500, 4500, 3500, 3500, 2000, 2000, 
                            2000, 100, 100]),
        "kv": np.array([450, 450, 350, 350, 200, 200, 200, 
                        10, 10]),
        "force_range": {
            "lower": np.array([-87, -87, -87, -87, -12, -12, 
                                -12, -100, -100]),
            "upper": np.array([87,  87,  87, 87,  12,  12, 
                                12,  100,  100]),
        },
    }

    

    utilities = {
        "target_x1": [0.70, -0.15, 1.5],
        "target_x2": [0.60, 0.15, 1.1],
        # "1": [0.60, 0.15, 1.5],
        # "2": [0.55, -0.2, 1.6],
        # "3": [0.5, -0.2, 1.2],
        # "4": [0.6, 0.15, 1.1],
        # "5": [0.7, 0.1, 1.1],
        # "6": [0.6, -0.1, 1.1],
    }

    env_infos = {
        "shelf": shelf_info,
        "base": base_info,
        "robot": robot_info,
        "utilities": utilities
    }

    return env_infos

def test_collision(env_infos):
    action1 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*2
    action2 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*8
    
    action = action1 + action2
    action = torch.tensor(action, dtype=torch.float32)

    return action



In [4]:
# Define CMA-ES parameters
num_guess = 5
popsize = 100
num_iterations = 10
# n_envs = num_guess * popsize
max_batch = MAX_BATCH
K = 2

# Define p2p motion
T = 1
dt = 0.01
n_joints = 7
basis = 'bs'
bounds = None
p2p_motion = Point2PointMotion(T=T, dt=dt, K=K, basis=basis, n=n_joints,
                            bounds=bounds, device=device)


In [5]:
# Generate the environment information
env_infos = env_infos_gen()

In [6]:
env = PandaShelf(n_envs=max_batch, gpu=True, use_gui=False,
                p2p_motion=p2p_motion)
env.load_world(env_infos)
# Define the initial_guesses
task_goal = env_infos["utilities"]["target_x2"]
x_goal = torch.tensor(task_goal).to(device)


[Genesis] [11:27:25] [WARNING] Overriding base link's pos with user provided value in morph.
[Genesis] [11:27:25] [WARNING] Overriding base link's quat with user provided value in morph.


In [ ]:
############################################################################
# Define the domain for discretization

n_joints=7
d0_theta = 20
dh_x = 0.05
d_theta_all = [d0_theta]*n_joints
d_theta = [int(d_theta_all[joint]) for joint in range(n_joints)]

# # define the robot
panda = PandaKinematics(device=device)

domain_decision = [torch.linspace(panda.theta_min[i],panda.theta_max[i],d_theta[i]).to(device) for i in range(n_joints)]

x_min = torch.tensor([0.45, -0.3, 0.8]).to(device)
x_max = torch.tensor([0.75, 0.3, 1.7]).to(device)

domain_task = [torch.linspace(x_min[i], x_max[i], int((x_max[i]-x_min[i])/dh_x)) for i in range(3)]
d_task = [len(domain_task[i]) for i in range(3)]


domain_task = [x.to(device) for x in domain_task]
domain_decision = [x.to(device) for x in domain_decision]

domain = domain_task + domain_decision


In [8]:
def objective_func(xs):
    """
    This function calculates the costs given the trajectories.
    Args:
        x: (num_guess*popsize, n_joints + K*n_joints) tensor
    Returns:
        costs: (num_guess*popsize,) tensor
    """
    global evaluate_time
    st_time = time.time()
    xs_pre = xs.clone()
    if xs.shape[0] < MAX_BATCH:
        xs = torch.cat([xs, torch.zeros((MAX_BATCH - xs.shape[0], xs.shape[1]), device=xs.device)], dim=0)

    x_goal = xs[:, :3]
    x = xs[:, 3:]

    env.reset(env_infos)
    robot = env_infos["robot"]["id"]
    theta_0 = robot.get_qpos()[:, :n_joints]

    """cost_collision"""
    
    # propage the joint configurations
    collisions = env.final_configuration(env_infos, x)
    # sum the collision flag at each time step as collision cost
    d_obst = torch.sum(collisions, dim=-1)

    """cost_control"""
    d_control = torch.linalg.norm(x - theta_0, dim=-1)

    """cost_goal"""
    # read robot's eef position
    eef = robot.get_link("grasping_target_hand")
    eef_pos = eef.get_pos()
    # print("x", x[:5])
    # print("eef_pos", eef_pos[:5])
    # print("eef position", eef_pos)
    d_goal = torch.linalg.norm(eef_pos - x_goal, dim=-1)
    d_total = 50*d_goal + 1*d_obst #+ 0.1*d_control

    if xs_pre.shape[0] < MAX_BATCH:
        d_total = d_total[:xs_pre.shape[0]]

    evaluate_time += time.time() - st_time
    return torch.exp(-(d_total))

In [9]:
model_folder = os.getcwd() + "/tt_models/"
file_name = 'panda_ik'
save_path = os.path.join(model_folder, file_name)
evaluate_time = 0
ttts = TTTS(func = objective_func, domain = domain, value_k=5, visit_k=5, 
            cross_max_iter=2, kickrank=20, max_tt=100, max_batch=MAX_BATCH,
        warm_start=True, verbose=False, save_path = save_path)#, test_mode=True)


The initial tt model is loaded!


In [10]:
ttts.aug_func_tt

10D TT tensor:

  6  12  15  20  20  20  20  20  20  20
  |   |   |   |   |   |   |   |   |   |
 (0) (1) (2) (3) (4) (5) (6) (7) (8) (9)
 / \ / \ / \ / \ / \ / \ / \ / \ / \ / \
1   6   50  50  50  50  50  50  50  20  1

In [11]:
task_goal = env_infos["utilities"]["target_x2"]
test_task = torch.tensor(task_goal)[None, :].to(device)

ttts.task_var = test_task

for _ in range(5):
    print("Iteration: ", _)
    if _ == 0:
        init_flag = True
    else:
        init_flag = False
    sol = ttts.iterate(max_iters = 3, param_C=3, init_flag=init_flag) # K x n
    sol_2 = ttts.parallel_cmaes_gpu(sol, num_iterations=50)


Iteration:  0
mcts-iters: 1, top-k values: [0.0006420978037189624, 0.00035871548126549887, 1.7812600116006007e-05, 7.944439918332726e-06, 4.281832446669877e-06]
mcts-iters: 2, top-k values: [0.0006420978037189624, 0.00035871548126549887, 1.7812600116006007e-05, 7.944439918332726e-06, 4.281832446669877e-06]
mcts-iters: 3, top-k values: [0.0006420978037189624, 0.00035871548126549887, 1.7812600116006007e-05, 7.944439918332726e-06, 4.281832446669877e-06]
Iteration:  1
mcts-iters: 1, top-k values: [0.0006420978037189624, 0.00035871548126549887, 1.7812600116006007e-05, 7.944439918332726e-06, 4.281832446669877e-06]
mcts-iters: 2, top-k values: [0.0006420978037189624, 0.00035871548126549887, 1.7812600116006007e-05, 7.944439918332726e-06, 4.281832446669877e-06]
mcts-iters: 3, top-k values: [0.0006420978037189624, 0.00035871548126549887, 1.7812600116006007e-05, 7.944439918332726e-06, 4.281832446669877e-06]
Iteration:  2
mcts-iters: 1, top-k values: [0.0006420978037189624, 0.00035871548126549887,

In [12]:
print("sol_2: ", sol_2.shape)

sol_2:  torch.Size([5, 7])


In [13]:
#save sol_2 as npy
torch.save(sol_2, "IK_sol_2.pth")
